# 1.Importação das bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
 
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# 2. carregamento do dataset

In [ ]:
#carregando o dataset
df = pd.read_csv('/kaggle/input/datasets/uciml/breast-cancer-wisconsin-data/data.csv')

#Mostra as 5 ptimeiras linhas 
df.head()

# 3.Conhecendo os dados 

In [ ]:
print("linhas e colunas:", df.shape) 

#informações sobre colunas e tipos de dados 
df.info()

# 4.Verificação de valores ausentes 

In [ ]:
#Verificar quantos valores ausentes existem em cada coluna
df.isnull().sum()

# 5. Limpeza dos dados 

In [ ]:
#Remove o identificador e a coluna vazia 
df = df.drop(columns=["id", "Unnamed: 32"])

df.head()

# 6. Distribuição dos diagnóstico

In [ ]:
# Quantidade de casos benignos e malignos
df["diagnosis"].value_counts () .plot(kind="bar")

plt.title("Distribuição dos diagnóstico")
plt.xlabel("Diagnóstico")
plt.ylabel("Quantidade")
plt.show()

# 7. Transfomação de variável alvo

In [ ]:
# Converte as classes para valores numéricos
df['diagnosis'] = df['diagnosis'].map({
'B': 0,
'M': 1
})

df['diagnosis'].value_counts()
 

# 8. Separação entre X e Y

In [ ]:
# X = características utilizadas para realizar a previsão
X = df.drop(columns='diagnosis')

# y = resposta que queremos prever
y = df['diagnosis']

print("X:", X.shape)
print("y:", y.shape)

# 9. Separação em treino e teste 

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
   X,
   y,
   test_size=0.20,
   random_state=42,
   stratify=y  # Mantém a proporção de benignos e malignos
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

# 10. Padronização dos dados 

In [ ]:
# Cria o padronizador
scaler = StandardScaler()

# Aprende a escala apenas com os dados de treino
X_train_scaled = scaler.fit_transform(X_train)

# Aplica a mesma transformação nos dados de teste
X_test_scaled = scaler.transform(X_test)

# 11. SVM com Kernel Linear

In [ ]:
# Criando o modelo SVM Linear
svm_linear = SVC(
   kernel='linear',
   C=1.0
)
# Treinamento
svm_linear.fit(X_train_scaled, y_train)
# Previsões
y_pred_linear = svm_linear.predict(X_test_scaled)

In [ ]:
acc_linear = accuracy_score(y_test, y_pred_linear)
print("Acurácia SVM Linear:", round(acc_linear, 4))
print("\nRelatório de classificação:")
print(classification_report(
   y_test,
   y_pred_linear,
   target_names=['Benigno', 'Maligno']
))

# 12. Matriz de confusão - Linear

In [ ]:
cm_linear = confusion_matrix(y_test, y_pred_linear)
disp = ConfusionMatrixDisplay(
   confusion_matrix=cm_linear,
   display_labels=['Benigno', 'Maligno']
)
disp.plot()
plt.title('Matriz de Confusão - SVM Linear')
plt.show()

# 13. Validação cruzada - Linear 

In [ ]:
pipeline_linear = Pipeline([
   ('scaler', StandardScaler()),
   ('svm', SVC(kernel='linear', C=1.0))
])
# Validação cruzada com 5 partes
scores_linear = cross_val_score(
   pipeline_linear,
   X,
   y,
   cv=5
)
print("Resultados:", scores_linear)
print("Média:", scores_linear.mean())

# 14. SVM com Kernel RBF

In [ ]:
# Criando o modelo SVM com kernel RBF
svm_rbf = SVC(
   kernel='rbf',
   C=1.0,
   gamma='scale'
)

# Treinamento
svm_rbf.fit(X_train_scaled, y_train)

# Previsões
y_pred_rbf = svm_rbf.predict(X_test_scaled)

# 15. Avaliação do SVM RBF

In [ ]:
acc_rbf = accuracy_score(y_test, y_pred_rbf)
print("Acurácia SVM RBF:", round(acc_rbf, 4))
print("\nRelatório de classificação:")
print(classification_report(
   y_test,
   y_pred_rbf,
   target_names=['Benigno', 'Maligno']
))

# 16. Matriz de confusão - RBF

In [ ]:
cm_rbf = confusion_matrix(y_test, y_pred_rbf)
disp = ConfusionMatrixDisplay(
   confusion_matrix=cm_rbf,
   display_labels=['Benigno', 'Maligno']
)
disp.plot()
plt.title('Matriz de Confusão - SVM RBF')
plt.show()

# 17. Validação cruzada - RBF

In [ ]:
pipeline_rbf = Pipeline([
   ('scaler', StandardScaler()),
   ('svm', SVC(kernel='rbf', C=1.0, gamma='scale'))
])
scores_rbf = cross_val_score(
   pipeline_rbf,
   X,
   y,
   cv=5
)
print("Resultados:", scores_rbf)
print("Média:", scores_rbf.mean())

# 18. Comparação dos modelos

In [ ]:
comparacao = pd.DataFrame({
   'Modelo': ['SVM Linear', 'SVM RBF'],
   'Acurácia Teste': [
       acc_linear,
       acc_rbf
   ],
   'Média Validação Cruzada': [
       scores_linear.mean(),
       scores_rbf.mean()
   ]
})
comparacao

# 19. Ajuste de C e gamma 

In [ ]:
# Pipeline evita problemas na padronização durante a busca
pipeline = Pipeline([
   ('scaler', StandardScaler()),
   ('svm', SVC())
])
parametros = [
   {
       'svm__kernel': ['linear'],
       'svm__C': [0.1, 1, 10]
   },
   {
       'svm__kernel': ['rbf'],
       'svm__C': [0.1, 1, 10],
       'svm__gamma': ['scale', 0.01, 0.1]
   }
]

# Testa diferentes combinações automaticamente
grid = GridSearchCV(
   pipeline,
   parametros,
   cv=5,
   scoring='accuracy'
)
grid.fit(X_train, y_train) 

# 20. melhores parametros encontrados

In [ ]:
print("melhores parâmetros:")
print(grid.best_params_)

print("melhor resultado:")
print(grid.best_score_)

# 21. avalição do melhor modelo

In [ ]:
# O GridSearch já guarda o melhor modelo encontrado
melhor_modelo = grid.best_estimator_
 
y_pred_melhor = melhor_modelo.predict(X_test)
 
print("Acurácia:", accuracy_score(y_test, y_pred_melhor))
 
print("\nRelatório de classificação:")
print(classification_report(
    y_test,
    y_pred_melhor,
    target_names=['Benigno', 'Maligno']
))

# 22. Matriz de confusão do melhor modelo

In [ ]:
cm = confusion_matrix(y_test, y_pred_melhor)
 
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Benigno', 'Maligno']
)
 
disp.plot()
plt.title('Matriz de Confusão - Melhor Modelo')
plt.show()